# 데이터 품질 점검 (2) — 결측과 값의 정합성

03 에서 테이블의 그레인과 키, 조인 규칙을 확정. 조인해도 행이 늘거나 표본이 조용히 빠지지 않는 상태.

남은 것은 값 자체의 문제. 컬럼의 결측 여부, 결측이라면 결함인지 아직 일어나지 않은 사건인지의 구분,
그리고 값들 사이의 정합성.

표본은 확정하지 않음. 어떤 주문을 쓸지는 검증할 가설에 따라 달라지므로,
**어떤 행이 어떤 이유로 어떤 분석에 쓸 수 없는지**만 정리.
가설이 정해지면 그 목록에서 골라 표본을 구성.

**확인 항목**

- 결측 — 9 개 테이블 전 컬럼의 결측 규모
- 결측의 성격 — 주문 상태로 설명되는 결측과 설명되지 않는 결측의 구분
- 값의 정합성 — 시각 컬럼의 순서가 뒤집힌 행
- 정리 — 조건별로 빠지는 행의 규모

## 0. 연결

리뷰 텍스트의 정제를 수행하므로 읽기 전용으로 열지 않음.

In [1]:
import duckdb

con = duckdb.connect('../olist.duckdb')

con

## 1. 결측 전수 점검

어떤 컬럼이 비어 있는지 9 개 테이블을 한 번에 확인.

DuckDB 의 `SUMMARIZE` 는 테이블의 컬럼별 요약을 반환하며 그중 `null_percentage` 가 결측률.
테이블마다 따로 실행하면 결과가 9 개로 흩어지므로, 결측이 있는 컬럼만 골라
`UNION ALL` 로 하나의 결과에 통합.

In [2]:
tables = [
    "orders",
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "product_category_name_translation",
    "geolocation",
]

blocks = [
    f"""
    SELECT '{t}' AS table_name,
           column_name,
           column_type,
           null_percentage
    FROM (SUMMARIZE {t})
    WHERE null_percentage > 0
    """
    for t in tables
]

sql = "UNION ALL".join(blocks) + "ORDER BY table_name, null_percentage DESC"

con.execute(sql).df()

,table_name,column_name,column_type,null_percentage
0,order_reviews,review_comment_title,VARCHAR,88.34
1,order_reviews,review_comment_message,VARCHAR,58.70
2,orders,order_delivered_customer_date,TIMESTAMP,2.98
3,orders,order_delivered_carrier_date,TIMESTAMP,1.79
4,orders,order_approved_at,TIMESTAMP,0.16
5,products,product_category_name,VARCHAR,1.85
6,products,product_name_lenght,BIGINT,1.85
7,products,product_description_lenght,BIGINT,1.85
8,products,product_photos_qty,BIGINT,1.85
9,products,product_weight_g,BIGINT,0.01


결측이 있는 테이블은 9 개 중 3 개. `customers`·`order_items`·`order_payments`·`sellers`·
번역표·`geolocation` 은 전 컬럼이 채워져 있음.

`review_score` 는 목록에 부재 — 리뷰 텍스트는 제목 88.34 %, 본문 58.70 % 가 비었으나
평점 자체는 결측 0. 리뷰를 남긴 주문이면 점수는 반드시 존재.

`products` 는 `product_category_name`·`product_name_lenght`·`product_description_lenght`·
`product_photos_qty` 넷이 모두 1.85 % 로 동일. 같은 행이 통째로 빈 경우인지 확인.

In [3]:
con.execute("""
SELECT COUNT(*)                                              AS product_cnt,
       COUNT(*) FILTER (WHERE product_category_name IS NULL) AS category_null,
       COUNT(*) FILTER (WHERE product_category_name      IS NULL
                          AND product_name_lenght        IS NULL
                          AND product_description_lenght IS NULL
                          AND product_photos_qty         IS NULL) AS four_cols_null,
       COUNT(*) FILTER (WHERE product_weight_g IS NULL)      AS weight_null
FROM products
""").df()

,product_cnt,category_null,four_cols_null,weight_null
0,32951,610,610,2


카테고리가 빈 610 건과 네 컬럼이 모두 빈 610 건이 같은 수. 서로 다른 결측 4 종이 아니라
상품 정보가 통째로 등록되지 않은 상품 610 개.

03 에서 확인한 번역표 누락 13 개와는 다른 문제. 그쪽은 카테고리명이 있으나 영문 번역이 없는
경우이고, 이쪽은 카테고리명 자체가 부재. 카테고리별 집계에서 610 개는 어느 카테고리에도
속하지 않으므로 `LEFT JOIN` 후 원문 폴백으로도 채워지지 않음.

무게·치수 결측 2 건은 규모가 달라 별도 취급.

남은 결측을 성격별로 구분.

리뷰 제목·본문은 선택 입력 항목이라 비어 있는 것이 값의 유실이 아니라 미입력.
다만 `NULL` 외에 빈 문자열로 들어온 행이 섞여 있으면 실제 미입력 규모가 달라지므로 확인.

In [4]:
con.execute("""
SELECT COUNT(*)                                                  AS review_cnt,
       COUNT(*) FILTER (WHERE review_comment_title   IS NULL)    AS title_null,
       COUNT(*) FILTER (WHERE TRIM(review_comment_title)   = '') AS title_blank,
       COUNT(*) FILTER (WHERE review_comment_message IS NULL)    AS message_null,
       COUNT(*) FILTER (WHERE TRIM(review_comment_message) = '') AS message_blank,
       COUNT(*) FILTER (WHERE review_comment_title   IS NULL
                          AND review_comment_message IS NULL)    AS both_null
FROM order_reviews
""").df()

,review_cnt,title_null,title_blank,message_null,message_blank,both_null
0,99224,87656,2,58247,9,56518


빈 문자열은 제목 2 건, 본문 9 건뿐. `NULL` 이 곧 미입력.

제목이 비었으나 본문은 있는 리뷰가 31,138 건(87,656 − 56,518),
반대로 본문이 비었으나 제목은 있는 리뷰가 1,729 건(58,247 − 56,518).
결측이 양방향으로 갈리므로 일괄 유실이 아니라 작성자가 항목별로 선택한 결과.

제목과 본문이 모두 없는 리뷰는 56,518 건으로 전체 99,224 건의 57 %.

빈 문자열 11 건은 `NULL` 과 같은 의미이므로 하나로 통일.
02 의 적재는 원본 그대로이므로 정제는 이 단계에서 수행.
정제 이후에는 위 결과의 빈 문자열이 0 이 되므로,
처음부터 재현하려면 02 를 먼저 실행해 원본 상태로 복원 필요.

In [5]:
con.execute("""
UPDATE order_reviews
SET review_comment_title   = CASE WHEN TRIM(review_comment_title)   = ''
                                  THEN NULL ELSE review_comment_title   END,
    review_comment_message = CASE WHEN TRIM(review_comment_message) = ''
                                  THEN NULL ELSE review_comment_message END
WHERE TRIM(review_comment_title)   = ''
   OR TRIM(review_comment_message) = ''
""").df()

,Count
0,11


11 행 변경. 제목 2 건과 본문 9 건이 서로 다른 행이라 합이 그대로 11.
이후 리뷰 텍스트의 미입력 여부는 `IS NULL` 만으로 판별 가능.

무게·치수 결측 2 건. 네 컬럼이 같은 행에서 비는지, 그리고 어떤 상품인지 확인.
`order_items` 와 연결해 실제 주문에 쓰인 상품인지도 함께 확인.

In [6]:
con.execute("""
SELECT p.product_id,
       p.product_category_name,
       p.product_weight_g,
       p.product_length_cm,
       p.product_height_cm,
       p.product_width_cm,
       COUNT(i.order_id) AS ordered_cnt
FROM products AS p
LEFT JOIN order_items AS i
       ON p.product_id = i.product_id
WHERE p.product_weight_g  IS NULL
   OR p.product_length_cm IS NULL
   OR p.product_height_cm IS NULL
   OR p.product_width_cm  IS NULL
GROUP BY ALL
""").df()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ordered_cnt
0,5eb564652db742ff8f28759cd8d2652a,NaN,<NA>,<NA>,<NA>,<NA>,17
1,09ff539a621711667c43eba6a3bd8466,bebes,<NA>,<NA>,<NA>,<NA>,1


두 상품 모두 무게·길이·높이·너비가 함께 결측. 치수 중 일부만 빠진 경우는 없음.

`5eb56465…` 는 카테고리도 비어 있어 앞서 확인한 610 개 미등록 상품에 포함되며, 17 회 주문.
`09ff539a…` 는 카테고리가 `bebes` 로 있고 1 회 주문.

영향 범위는 `order_items` 112,650 행 중 18 행. 별도 보정 없이 진행 가능.

## 2. 날짜 결측의 성격

1 에서 `orders` 의 시각 컬럼 세 개에 결측을 확인 —
승인 0.16 %, 운송사 인계 1.79 %, 배송 완료 2.98 %.

비율만 보면 배송 완료일 2,965 건이 가장 큰 문제로 보이나, 이 컬럼은 사건이 일어나야 채워지는 값.
아직 배송 중인 주문이라면 비어 있는 것이 정상이고 결함이 아님.
반대로 `delivered` 인데 완료일이 없다면 상태와 값이 모순.

`order_status` 와 교차해 상태로 설명되는 결측과 설명되지 않는 결측을 구분.

In [7]:
con.execute("""
SELECT order_status,
       COUNT(*) AS order_cnt,
       COUNT(*) FILTER (WHERE order_approved_at             IS NULL) AS approved_null,
       COUNT(*) FILTER (WHERE order_delivered_carrier_date  IS NULL) AS carrier_null,
       COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS delivered_null
FROM orders
GROUP BY order_status
ORDER BY order_cnt DESC
""").df()

,order_status,order_cnt,approved_null,carrier_null,delivered_null
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


상태별로 결측이 갈리는 모습이 뚜렷.

`unavailable` 609 건, `invoiced` 314 건, `processing` 301 건은 출고 자체가 없어
인계일과 완료일이 함께 결측. `created` 5 건은 승인조차 되지 않은 상태.
해당 단계에 도달하지 않아 값이 없는 경우.

`shipped` 1,107 건은 운송사 인계까지 끝나고 완료일만 결측.
아직 도착하지 않은 주문이라면 설명이 되나, 그러려면 구매 시기가 수집 종료 무렵이어야 함.
시기는 아래에서 확인.

`canceled` 625 건은 취소 시점이 제각각이라 결측이 혼재 —
승인 전 취소 141 건, 출고 전 취소 550 건. 배송 완료일이 있는 주문도 6 건 존재.

상태와 정면으로 모순되는 것은 `delivered` 인데 완료일이 없는 8 건. 따로 확인.

In [8]:
con.execute("""
SELECT o.order_id,
       o.order_purchase_timestamp,
       o.order_approved_at,
       o.order_delivered_carrier_date,
       o.order_estimated_delivery_date,
       COALESCE(r.review_cnt, 0) AS review_cnt,
       r.max_score
FROM orders AS o
LEFT JOIN (
    SELECT order_id,
           COUNT(*)          AS review_cnt,
           MAX(review_score) AS max_score
    FROM order_reviews
    GROUP BY order_id
) AS r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NULL
ORDER BY o.order_purchase_timestamp
""").df()

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_estimated_delivery_date,review_cnt,max_score
0,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,2017-06-23,1,5
1,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,2017-12-18,1,5
2,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,2018-06-26,1,1
3,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,2018-07-16,1,5
4,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,2018-07-19,1,5
5,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,2018-07-30,1,5
6,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,2018-07-24,1,5
7,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,2018-07-30,1,5


8 건 모두 리뷰가 1 건씩 달려 있고 그중 7 건이 5 점.

`2d858f45…` 만 운송사 인계일도 비어 있고, 나머지 7 건은 승인과 인계까지 기록된 뒤
완료 시각만 없음.

2018-07-01 구매가 3 건이고 그중 두 건은 인계 시각까지 `2018-07-03 13:57:00` 으로 동일.
무작위 누락이 아니라 특정 배치에서 완료 기록이 함께 빠진 것으로 보임.

평점만으로는 상품을 실제로 받았는지 단정할 수 없음. 1 점인 `ab7c89dc…` 는 받지 못했을
가능성이 있고, 5 점인 건도 배송이 아닌 다른 것을 평가했을 수 있음. 리뷰 본문을 직접 확인.

In [9]:
rows = con.execute("""
SELECT o.order_id,
       r.review_score,
       r.review_comment_title,
       r.review_comment_message
FROM orders AS o
JOIN order_reviews AS r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NULL
ORDER BY r.review_score, o.order_purchase_timestamp
""").fetchall()

for order_id, score, title, message in rows:
    print(f"[{score}점] {order_id}")
    print(f"  제목: {title}")
    print(f"  본문: {message}")
    print()

[1점] ab7c89dc1bf4a1ead9d6ec1ec8968a84
  제목: Péssimo
  본문: Comprei um produto de uma marca e recebi outro inferior ao que comprei sem falar que me cobraram frete é tive que pegar no correio

[5점] 2d858f451373b04fb5c984a1cc2defaf
  제목: None
  본문: None

[5점] 2d1e2d5bf4dc7227b3bfebb81328c15f
  제목: None
  본문: Chegou rápido tudo ok

[5점] f5dd62b788049ad9fc0526e3ad11a097
  제목: Entrega super rápida.
  본문: Produto novo, muito bom.

[5점] 20edc82cf5400ce95e1afacc25798b31
  제목: Muito bom
  본문: Adorei

[5점] 2ebdfc4f15f23b91474edf87475f108e
  제목: None
  본문: None

[5점] 0d3268bad9b086af767785e3f0fc0133
  제목: Excelente!
  본문: O produto chegou muito antes do prazo previsto, veio em ótimas condições e muito bem embalado. Recomendo!

[5점] e69f75a717d64fc5ecdfae42b2e8e086
  제목: None
  본문: None



본문이 있는 5 건 모두 상품을 받았음을 나타냄.

1 점인 `ab7c89dc…` 의 본문은 "산 것과 다른, 그보다 못한 상품을 받았고 운임까지 청구됐으며
우체국에서 직접 찾아와야 했다". 받지 못한 것이 아니라 다른 상품을 받은 것.

5 점 중 `0d3268ba…` 는 "예정일보다 훨씬 일찍 도착했고 상태와 포장도 좋았다",
`2d1e2d5b…` 는 "빨리 도착했고 다 괜찮다", `f5dd62b7…` 는 제목이 "배송 매우 빠름".
시스템에는 완료 시각이 없는데 고객은 도착을 명시.
나머지 3 건은 본문 없이 5 점만 남김.

8 건 모두 배송은 실제로 이루어졌고 완료 시각만 기록되지 않음.
`ab7c89dc…` 의 "우체국에서 직접 수령"은 배송원이 전달하지 않은 경우 완료 시각이 남지 않을
가능성을 시사하나, 1 건이므로 원인으로 일반화하지 않음.

배송이 되었더라도 완료 시각이 없으면 소요 시간을 계산할 수 없음.
배송 소요 시간을 쓰는 분석에서는 사용 불가.

반대 방향도 확인. `canceled` 인데 배송 완료일이 있는 6 건이
배송이 끝난 뒤 취소된 것인지 확인.

In [10]:
con.execute("""
SELECT o.order_id,
       o.order_purchase_timestamp,
       o.order_delivered_customer_date,
       date_diff('day', o.order_purchase_timestamp,
                        o.order_delivered_customer_date) AS delivery_days,
       COALESCE(r.review_cnt, 0) AS review_cnt,
       r.max_score
FROM orders AS o
LEFT JOIN (
    SELECT order_id,
           COUNT(*)          AS review_cnt,
           MAX(review_score) AS max_score
    FROM order_reviews
    GROUP BY order_id
) AS r ON o.order_id = r.order_id
WHERE o.order_status = 'canceled'
  AND o.order_delivered_customer_date IS NOT NULL
ORDER BY o.order_purchase_timestamp
""").df()

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days,review_cnt,max_score
0,65d1e226dfaeb8cdc42f665422522d14,2016-10-03 21:01:41,2016-11-08 10:58:34,36,1,1
1,770d331c84e5b214bd9dc70a10b829d0,2016-10-07 14:52:30,2016-10-14 15:07:11,7,1,1
2,8beb59392e21af5eb9547ae1a9938d06,2016-10-08 20:17:50,2016-10-19 18:47:43,11,1,1
3,dabf2b0e35b423f94618bf965fcb7514,2016-10-09 00:56:52,2016-10-16 14:36:59,7,1,5
4,2c45c33d2f9cb8ff8b1c86cc28c11c30,2016-10-09 15:39:56,2016-11-09 14:53:50,31,1,1
5,1950d777989f6a877539f53795b4c3c3,2018-02-19 19:48:52,2018-03-21 22:03:51,30,1,3


6 건 중 5 건이 2016-10-03 ~ 10-09 일주일 사이에 밀집하고 나머지 1 건만 2018-02.
03 에서 확인한 결제 기록 없는 주문 1 건도 2016-09. 기록 결함이 데이터셋 초기에 집중.

평점은 1 점이 4 건. 배송은 되었으나 이후 취소로 처리된 주문이며 고객 평가도 낮다.
`delivered` 만 보는 분석에서는 이 6 건이 자동으로 제외.

앞에서 `shipped` 1,107 건을 아직 도착하지 않은 주문으로 보았으나 확인한 것은 아님.
그렇게 보려면 데이터 수집이 끝나는 시점 근처에 밀집해야 함.
한참 전에 구매된 주문이 아직 `shipped` 라면 배송이 진행 중인 것이 아니라
기록이 멈춘 것이므로 다른 취급이 필요.

월별 주문 규모와 상태 분포를 함께 확인. 데이터셋의 시작과 끝이 온전한지,
`shipped` 가 어느 시기에 놓여 있는지 한 번에 확인.

In [11]:
con.execute("""
SELECT date_trunc('month', order_purchase_timestamp)      AS month,
       COUNT(*)                                           AS order_cnt,
       COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered,
       COUNT(*) FILTER (WHERE order_status = 'shipped')   AS shipped,
       COUNT(*) FILTER (WHERE order_status NOT IN ('delivered', 'shipped'))
                                                          AS other_status
FROM orders
GROUP BY month
ORDER BY month
""").df()

,month,order_cnt,delivered,shipped,other_status
0,2016-09-01,4,1,1,2
1,2016-10-01,324,265,8,51
2,2016-12-01,1,1,0,0
3,2017-01-01,800,750,16,34
4,2017-02-01,1780,1653,21,106
5,2017-03-01,2682,2546,45,91
6,2017-04-01,2404,2303,49,52
7,2017-05-01,3700,3546,55,99
8,2017-06-01,3245,3135,47,63
9,2017-07-01,4026,3872,56,98


데이터셋의 양 끝이 잘려 있음. 2016-09 가 4 건, 2016-10 이 324 건,
2016-11 은 행 자체가 없고 2016-12 가 1 건. 2017-01 부터 800 건으로 올라서며 궤도에 오름.
끝도 2018-09 가 16 건, 2018-10 이 4 건이며 두 달 모두 `delivered` 가 0.

온전한 구간은 2017-01 ~ 2018-08. 그 밖의 349 건은 전체 99,441 건의 0.35 %.

`shipped` 는 수집 종료 무렵에 몰려 있지 않음. 2018-03 이 133 건으로 가장 많고
2018-04 99 건, 2017-11 72 건 등 전 기간에 퍼져 있으며 마지막 두 달은 1 건과 0 건.
구매 후 여러 달이 지나도록 `shipped` 에 머문 주문이라는 뜻이므로
아직 도착하지 않은 주문으로 볼 수 없음. 배송이 끝나지 않았거나 완료 기록이 남지 않은 쪽.

어느 쪽이든 완료 시각이 없어 배송 소요 시간을 쓰는 분석에서는 사용 불가.

## 3. 시각 순서의 정합성

`orders` 는 구매·승인·운송사 인계·고객 수령·예상 배송의 다섯 시각을 보유.
앞의 넷은 실제로 일어난 순서가 있으므로 뒤집힌 행이 있으면 값의 오류.

배송 소요 시간을 `고객 수령 − 구매` 로 계산할 예정이므로, 그 둘이 뒤집힌 행이 있으면 음수가 발생.
계산식을 쓰기 전에 확인.

중간 시각까지 함께 보는 것은 뒤집힌 행이 많을 경우 시각 기록 전반의 신뢰도를
의심해야 하기 때문. 값이 채워진 행만 대상이므로 배송이 완료된 주문으로 한정.

In [12]:
con.execute("""
SELECT COUNT(*) AS delivered_with_date,
       COUNT(*) FILTER (WHERE order_approved_at < order_purchase_timestamp)
           AS approved_before_purchase,
       COUNT(*) FILTER (WHERE order_delivered_carrier_date < order_approved_at)
           AS carrier_before_approved,
       COUNT(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date)
           AS customer_before_carrier,
       COUNT(*) FILTER (WHERE order_delivered_customer_date < order_purchase_timestamp)
           AS customer_before_purchase
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
""").df()

,delivered_with_date,approved_before_purchase,carrier_before_approved,customer_before_carrier,customer_before_purchase
0,96470,0,1350,23,0


구매보다 승인이 앞선 행 0 건, 구매보다 수령이 앞선 행 0 건.
배송 소요 시간을 `고객 수령 − 구매` 로 계산해도 음수가 나오지 않음.

중간 시각에는 뒤집힌 행이 존재. 승인보다 인계가 앞선 행 1,350 건(1.4 %),
인계보다 수령이 앞선 행 23 건. 뒤집힌 폭을 확인.

In [13]:
con.execute("""
SELECT gap_type,
       COUNT(*)                                AS rows,
       COUNT(*) FILTER (WHERE gap_hours <= 1)  AS within_1h,
       COUNT(*) FILTER (WHERE gap_hours <= 24) AS within_1d,
       COUNT(*) FILTER (WHERE gap_hours >  24) AS over_1d,
       MAX(gap_hours)                          AS max_hours
FROM (
    SELECT '1. 승인 앞 인계' AS gap_type,
           date_diff('hour', order_delivered_carrier_date,
                             order_approved_at) AS gap_hours
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_delivered_carrier_date < order_approved_at

    UNION ALL

    SELECT '2. 인계 앞 수령',
           date_diff('hour', order_delivered_customer_date,
                             order_delivered_carrier_date)
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_delivered_customer_date < order_delivered_carrier_date
)
GROUP BY gap_type
ORDER BY gap_type
""").df()

,gap_type,rows,within_1h,within_1d,over_1d,max_hours
0,1. 승인 앞 인계,1350,335,923,427,4109
1,2. 인계 앞 수령,23,2,7,16,386


폭이 1 시간 이내부터 171 일까지 제각각.
승인과 인계 시각이 정확하게 기록되지 않았다는 정도로 정리.
대충 입력되었거나 배송이 정해진 순서대로 진행되지 않은 경우로 보이며,
고객이 겪는 값이 아니므로 원인은 더 파지 않음.

구매와 수령은 역전이 없어 총 소요 시간에는 영향 없음.
구간을 나눠 계산하는 분석에서만 1,373 건이 걸리며 그때 제외.

## 4. 값의 도메인

결측이 아니라 채워져 있는 값이 가능한 범위 안에 있는지 확인.
평점이 6 이거나 금액이 음수이면 결측과는 다른 종류의 문제.

1 에서 쓴 `SUMMARIZE` 가 결측률과 함께 컬럼별 최솟값·최댓값을 반환하므로 같은 함수를 재사용.
숫자와 시각 컬럼만 대상으로 하며, 문자열은 최솟값·최댓값이 의미를 갖지 않아 제외.
`geolocation` 은 사용하지 않으므로 제외.

무엇을 이상값으로 볼지는 미리 정하지 않음. 0 원은 무료 증정품일 수 있고 할부 0 은
일시불 표기일 수 있음. 범위를 먼저 보고 판단.
`review_score` 만 1 ~ 5 로 도메인이 정해져 있어 밖으로 나가면 오류.

In [14]:
value_tables = [
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
]

blocks = [
    f"""
    SELECT '{t}' AS table_name,
           column_name,
           column_type,
           min,
           max
    FROM (SUMMARIZE {t})
    WHERE column_type IN ('BIGINT', 'DOUBLE', 'TIMESTAMP')
    """
    for t in value_tables
]

sql = "UNION ALL".join(blocks) + "ORDER BY table_name, column_name"

con.execute(sql).df()

,table_name,column_name,column_type,min,max
0,order_items,freight_value,DOUBLE,0.0,409.68
1,order_items,order_item_id,BIGINT,1,21
2,order_items,price,DOUBLE,0.85,6735.0
3,order_items,shipping_limit_date,TIMESTAMP,2016-09-19 00:15:34,2020-04-09 22:35:08
4,order_payments,payment_installments,BIGINT,0,24
5,order_payments,payment_sequential,BIGINT,1,29
6,order_payments,payment_value,DOUBLE,0.0,13664.08
7,order_reviews,review_answer_timestamp,TIMESTAMP,2016-10-07 18:32:28,2018-10-29 12:27:35
8,order_reviews,review_creation_date,TIMESTAMP,2016-10-02 00:00:00,2018-08-31 00:00:00
9,order_reviews,review_score,BIGINT,1,5


`review_score` 는 1 ~ 5 로 도메인 안. 범위를 벗어난 평점 없음.
`order_item_id` 최대 21 은 03 에서 확인한 21 행짜리 주문과 일치.
상품의 치수와 길이 컬럼은 모두 양수.

눈에 띄는 것 넷.

- `shipping_limit_date` 최댓값이 2020-04-09. 다른 시각 컬럼은 모두 2018-11 이전인데 이것만 2020 년
- `freight_value` 와 `payment_value` 의 최솟값이 0
- `payment_installments` 최솟값이 0
- `product_weight_g` 최솟값이 0

0 은 무료 배송이나 전액 바우처 결제처럼 정상일 수 있으므로 규모부터 확인.

In [15]:
con.execute("""
SELECT '1. shipping_limit_date 가 2019 년 이후' AS item,
       COUNT(*)                                 AS rows
FROM order_items
WHERE shipping_limit_date >= DATE '2019-01-01'

UNION ALL
SELECT '2. freight_value = 0', COUNT(*)
FROM order_items
WHERE freight_value = 0

UNION ALL
SELECT '3. payment_value = 0', COUNT(*)
FROM order_payments
WHERE payment_value = 0

UNION ALL
SELECT '4. payment_installments = 0', COUNT(*)
FROM order_payments
WHERE payment_installments = 0

UNION ALL
SELECT '5. product_weight_g = 0', COUNT(*)
FROM products
WHERE product_weight_g = 0

ORDER BY item
""").df()

,item,rows
0,1. shipping_limit_date 가 2019 년 이후,4
1,2. freight_value = 0,383
2,3. payment_value = 0,9
3,4. payment_installments = 0,2
4,5. product_weight_g = 0,4


전부 소수. 가장 많은 것이 `freight_value` 0 인 383 건으로 `order_items` 112,650 행의 0.34 %.
운임을 받지 않은 주문으로 볼 수 있어 값이 잘못된 것으로 보기 어려움.

`payment_value` 0 이 9 건, `payment_installments` 0 이 2 건, `product_weight_g` 0 이 4 건.
어느 쪽도 집계에 영향을 줄 규모가 아님.

`shipping_limit_date` 가 2019 년 이후인 4 건은 수집 기간 밖이라 값이 잘못된 것으로 보임.
다만 이 컬럼은 판매자의 발송 기한이고 배송 소요 시간 계산에 쓰지 않으므로 지금은 영향 없음.
이 컬럼을 쓰는 분석에서 제외.

## 5. 정리

확인한 내용을 모으고, 조건별로 빠지는 행의 규모를 집계.
표본을 확정하지 않으므로 조건을 조합하지 않고 각각의 규모만 제시.

리뷰 없음은 배송이 완료된 주문으로 한정. 취소되었거나 상품을 확보하지 못한 주문은
리뷰가 없는 것이 자연스러우므로, 있을 만한데 없는 경우만 집계.

In [16]:
con.execute("""
SELECT '1. 전체 주문' AS condition,
       COUNT(*)       AS rows
FROM orders

UNION ALL
SELECT '2. order_status 가 delivered 아님', COUNT(*)
FROM orders
WHERE order_status <> 'delivered'

UNION ALL
SELECT '3. delivered 이나 배송 완료일 없음', COUNT(*)
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NULL

UNION ALL
SELECT '4. 배송 완료 주문 중 리뷰 없음', COUNT(*)
FROM orders AS o
LEFT JOIN (SELECT DISTINCT order_id FROM order_reviews) AS r
       ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND r.order_id IS NULL

UNION ALL
SELECT '5. 품목 기록 없음', COUNT(*)
FROM orders AS o
LEFT JOIN (SELECT DISTINCT order_id FROM order_items) AS i
       ON o.order_id = i.order_id
WHERE i.order_id IS NULL

UNION ALL
SELECT '6. 온전한 기간(2017-01 ~ 2018-08) 밖', COUNT(*)
FROM orders
WHERE order_purchase_timestamp <  DATE '2017-01-01'
   OR order_purchase_timestamp >= DATE '2018-09-01'

ORDER BY condition
""").df()

,condition,rows
0,1. 전체 주문,99441
1,2. order_status 가 delivered 아님,2963
2,3. delivered 이나 배송 완료일 없음,8
3,4. 배송 완료 주문 중 리뷰 없음,646
4,5. 품목 기록 없음,775
5,6. 온전한 기간(2017-01 ~ 2018-08) 밖,349


배송 완료 주문 중 리뷰가 없는 것은 646 건으로 96,470 건의 0.67 %.
전체 주문 기준으로 세면 768 건이나, 그중 122 건은 배송이 완료되지 않아
애초에 리뷰를 기대할 수 없는 주문.

조건들은 기준 범위가 서로 다르고 겹치기도 하므로 합산해서 쓰지 않음.
품목 기록이 없는 775 건은 03 에서 확인한 대로 `delivered` 가 하나도 없어 조건 2 에 포함.

**확인한 것**

| 항목 | 내용 |
|---|---|
| 결측 있는 테이블 | 9 개 중 3 개 — `orders`, `order_reviews`, `products` |
| 리뷰 텍스트 | 제목 88.34 %, 본문 58.70 % 미입력. 평점은 결측 0 |
| 상품 정보 | 610 개가 카테고리·이름 길이·설명 길이·사진 수 통째로 미등록 |
| 무게·치수 | 2 개 상품에서 네 컬럼이 함께 결측. `order_items` 18 행 |
| 날짜 결측 | 대부분 주문 상태로 설명됨. `delivered` 인데 완료일 없는 8 건만 모순 |
| 시각 순서 | 구매·수령 역전 0 건. 중간 시각은 1,373 건이 뒤집혀 있음 |
| 기간 | 온전한 구간은 2017-01 ~ 2018-08. 밖은 349 건 |
| 값의 도메인 | 평점은 1 ~ 5 로 정상. 0 값과 범위 밖 시각이 있으나 가장 큰 것이 운임 0 인 383 행 |

**정제한 것**

- `order_reviews` 의 빈 문자열 11 건을 `NULL` 로 변환.
  이후 리뷰 텍스트의 미입력 여부는 `IS NULL` 만으로 판별.
  02 는 원본을 그대로 적재하므로, 02 를 다시 실행하면 04 도 재실행 필요.

**넘길 것**

- 리뷰가 여러 건인 주문 547 건의 처리 기준 — 주문 단위 평점을 만들 때마다 걸림.
  03 에서 평점이 갈리는 주문이 202 건임을 확인했고, 평균·최신·최저 중 무엇으로 정할지는 미결
- 지역 단위를 주로 둘지 도시로 내릴지 — 03 에서 주 27 개, `(도시, 주)` 4,310 개를 확인했으나
  집계에 쓸 만한 규모인지는 미확인
- 상품을 받지 못한 주문 756 건에 달린 리뷰 — 낮은 평점의 원인을 볼 때 별도 관찰 대상
- 무게·부피가 배송에 미치는 영향 — `products` 의 무게·치수와 `order_items` 의 운임으로 확인 가능

## 6. 연결 종료

DuckDB 파일은 한 프로세스만 쓰기 모드로 열 수 있으므로 다음 노트북을 위해 여기서 종료.

In [17]:
con.close()